In [1]:
import sys
import pandas as pd
import os
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')
import locale
locale.setlocale(locale.LC_ALL, 'en_IN.utf8')
import pyodbc
from sqlalchemy import create_engine



In [1]:
def get_today_files(base_folder):
    from datetime import datetime, date
    import os

    today = date.today()
    valid_files = []

    # -------------------------------
    # TARGET MONTH FOLDER (Apr'26)
    # -------------------------------
    folder_name = datetime.today().strftime("%b'%y")
    target_folder = os.path.join(base_folder, folder_name)
    print(folder_name)
    print(target_folder)


    if not os.path.exists(target_folder):
        print(f"Folder not found: {target_folder}")
        return []

    # -------------------------------
    # ONLY CHANGE HERE (no os.walk)
    # -------------------------------

    # print(os.listdir(target_folder))
    for file in os.listdir(target_folder):
        if file.endswith((".xlsx", ".xlsb")):
            file_path = os.path.join(target_folder, file)
            

            try:
                
                modified_time = datetime.fromtimestamp(
                    os.path.getmtime(file_path)
                ).date()

                print(modified_time)

                if modified_time == today:
                    valid_files.append(file_path)
                    # print(file_path)


            except Exception as e:
                print(f"Skipping file (time error): {file_path}")

    return valid_files

In [2]:
base_folder = r"\\10.0.11.37\bd_pbp\DB Upload\Revenue_cal_table Update"
valid_files=get_today_files(base_folder)
valid_files

Nov'26
\\10.0.11.37\bd_pbp\DB Upload\Revenue_cal_table Update\Nov'26
2026-11-06
2026-11-06
2026-11-12
2026-11-10
2026-11-13
2026-11-10
2026-11-10
2026-11-11
2026-11-10
2026-11-12
2026-11-12
2026-11-12
2026-11-03
2026-11-11
2026-11-13
2026-11-04
2026-11-07
2026-11-06
2026-11-05
2026-11-16
2026-11-02
2026-11-11
2026-11-02
2026-11-09
2026-11-13
2026-11-01
2026-11-02


[]

In [11]:
correct_cols = [
    "Leadid","BookingDate","FY_Year","BookingMonth",
    "IssuanceDate","RiskStartDate","RiskEndDate",
    "PolicyNo","InsurerName","BU","Product","SubProduct",
    "VehicleSubClass","PlanType","PlanName","PolicyStatus",
    "ODPremium","TPPremium","NetPremium","TotalPremium",
    "PointOn","Points","IRDA","Reward","OtherDues","FinalPayin",
    "Remarks","Status","ReceivedMonth","FirstYrAmt","FifthYrAmt",
    "Additional_Rec","Total_Rec","Diff_Amt",
    "RewardOD","RewardTP","IncentiveOD","IncentiveTP"
]  


expected_cols = {
                col: col.strip().lower()
                for col in correct_cols
                             }

print(expected_cols)
b=set(expected_cols)
print(b)
print(len(b))


{'Leadid': 'leadid', 'BookingDate': 'bookingdate', 'FY_Year': 'fy_year', 'BookingMonth': 'bookingmonth', 'IssuanceDate': 'issuancedate', 'RiskStartDate': 'riskstartdate', 'RiskEndDate': 'riskenddate', 'PolicyNo': 'policyno', 'InsurerName': 'insurername', 'BU': 'bu', 'Product': 'product', 'SubProduct': 'subproduct', 'VehicleSubClass': 'vehiclesubclass', 'PlanType': 'plantype', 'PlanName': 'planname', 'PolicyStatus': 'policystatus', 'ODPremium': 'odpremium', 'TPPremium': 'tppremium', 'NetPremium': 'netpremium', 'TotalPremium': 'totalpremium', 'PointOn': 'pointon', 'Points': 'points', 'IRDA': 'irda', 'Reward': 'reward', 'OtherDues': 'otherdues', 'FinalPayin': 'finalpayin', 'Remarks': 'remarks', 'Status': 'status', 'ReceivedMonth': 'receivedmonth', 'FirstYrAmt': 'firstyramt', 'FifthYrAmt': 'fifthyramt', 'Additional_Rec': 'additional_rec', 'Total_Rec': 'total_rec', 'Diff_Amt': 'diff_amt', 'RewardOD': 'rewardod', 'RewardTP': 'rewardtp', 'IncentiveOD': 'incentiveod', 'IncentiveTP': 'incentive

In [ ]:
import sys
import pandas as pd
import os
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')
import locale
locale.setlocale(locale.LC_ALL, 'en_IN.utf8')
import pyodbc
from sqlalchemy import create_engine



def get_today_files(base_folder):
    from datetime import datetime, date
    import os

    today = date.today()
    valid_files = []

    # -------------------------------
    # TARGET MONTH FOLDER (Apr'26)
    # -------------------------------
    folder_name = datetime.today().strftime("%b'%y")
    target_folder = os.path.join(base_folder, folder_name)
    print(folder_name)
    print(target_folder)


    if not os.path.exists(target_folder):
        print(f"Folder not found: {target_folder}")
        return []

    # -------------------------------
    # ONLY CHANGE HERE (no os.walk)
    # -------------------------------

    print(os.listdir(target_folder))
    for file in os.listdir(target_folder):
        if file.endswith((".xlsx", ".xlsb")):
            file_path = os.path.join(target_folder, file)
            # print(file_path)

            try:
                
                modified_time = datetime.fromtimestamp(
                    os.path.getmtime(file_path)
                ).date()
                # print(modified_time)
                if modified_time == today:
                    valid_files.append(file_path)

            except Exception as e:
                print(f"Skipping file: {file_path}")
                print(f"Actual error: {e}")

    return valid_files




# if  valid_files:
#     print("files found today. Exiting.")
# else:
#     print('no files there')    
# print(valid_files)python test.py
# for x in valid_files:
#     print(x)




# -------------------------------
def ingest_files(valid_files, log_path, mapping_path):

    dfs = []
    log_data = []
    today_str = datetime.today().strftime("%Y-%m-%d")

    # -------------------------------
    # REQUIRED SCHEMA
    # -------------------------------
    correct_cols = [
        "Leadid","BookingDate","FY_Year","BookingMonth",
        "IssuanceDate","RiskStartDate","RiskEndDate",
        "PolicyNo","InsurerName","BU","Product","SubProduct",
        "VehicleSubClass","PlanType","PlanName","PolicyStatus",
        "ODPremium","TPPremium","NetPremium","TotalPremium",
        "PointOn","Points","IRDA","Reward","OtherDues","FinalPayin",
        "Remarks","Status","ReceivedMonth","FirstYrAmt","FifthYrAmt",
        "Additional_Rec","Total_Rec","Diff_Amt",
        "RewardOD","RewardTP","IncentiveOD","IncentiveTP"
    ]

    date_cols = ["BookingDate","IssuanceDate","RiskStartDate","RiskEndDate"]

    # -------------------------------
    # GET FILES
    # -------------------------------
    file_list = valid_files

    print(f" Total today's files: {len(file_list)}")

    # -------------------------------
    # LOOP FILES
    # -------------------------------
    for file_path in file_list:

        file = os.path.basename(file_path)
        print()
        print(f"Reading: {file_path}")

        status = "Success"
        error_msg = None
        row_count = 0

        try:
            # -------------------------------
            # READ FILE
            # -------------------------------
            if file.endswith(".xlsb"):
                temp_df = pd.read_excel(
                    file_path,
                    engine="pyxlsb",
                    sheet_name=0,
                    keep_default_na=False
                )

                # XLSB DATE FIX
                for col in date_cols:
                    if col in temp_df.columns:
                        temp_df[col] = temp_df[col].apply(
                            lambda x: pd.to_datetime(
                                x, origin="1899-12-30", unit="D", errors="coerce"
                            ) if isinstance(x, (int, float))
                            else pd.to_datetime(x, errors="coerce")
                        )

            else:
                temp_df = pd.read_excel(
                    file_path,
                    engine="openpyxl",
                    sheet_name=0,
                    keep_default_na=False
                )
                
                for col in date_cols:
                    if col in temp_df.columns:
                        temp_df[col] = temp_df[col].apply(
                            lambda x: pd.to_datetime(
                                x, origin="1899-12-30", unit="D", errors="coerce"
                            ) if isinstance(x, (int, float))
                            else pd.to_datetime(x, errors="coerce")
                        )

            

            # -------------------------------
            # CLEAN COLUMNS
            # -------------------------------
            temp_df.columns = temp_df.columns.str.strip()

            temp_df = temp_df.loc[
            :,
            ~temp_df.columns.str.contains("^Unnamed", case=False, na=False)
            ]

            # -------------------------------
            # SCHEMA VALIDATION
            # -------------------------------
            file_cols = {
                            col.strip().lower(): col
                            for col in temp_df.columns
                        }
            
            # Expected columns: lowercase version -> standard column name
            expected_cols = {
                col.strip().lower(): col
                for col in correct_cols
                             }

            missing_cols = list(set(expected_cols) - set(file_cols))
            extra_cols = list(set(file_cols) - set(expected_cols))

            if missing_cols or extra_cols:
                status = "Failed"
                error_msg = f"Missing: {missing_cols} | Extra: {extra_cols}"

            else:
                # Convert whatever casing is present
                # into your standard column names
                rename_mapping = {
                    file_cols[key]: expected_cols[key]
                    for key in expected_cols
                }

                print(rename_mapping)
            
                temp_df = temp_df.rename(columns=rename_mapping)
            
                # Enforce standard column order
                temp_df = temp_df[correct_cols]
            
                row_count = len(temp_df)
            
                dfs.append(temp_df)

        except Exception as e:
            status = "Failed"
            error_msg = str(e)

        # -------------------------------
        # LOG ENTRY (ALWAYS LOG)
        # -------------------------------
        log_data.append({
            "Date": today_str,
            "FileName": file,
            "FilePath": file_path,
            "Status": status,
            "Rows": row_count,
            "Error": error_msg
        })

    # -------------------------------
    # FINAL DF
    # -------------------------------
    final_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    print(f" Final rows appended: {len(final_df)}")

    # -------------------------------
    # SAVE LOG (APPEND MODE)
    # -------------------------------
    log_df = pd.DataFrame(log_data)

    try:
        if os.path.exists(log_path):
            old_log = pd.read_excel(log_path)
            log_df = pd.concat([old_log, log_df], ignore_index=True)

        log_df.to_excel(log_path, index=False)
        print(f" Log saved: {log_path}")

    except Exception as e:
        print(" Log file is open/locked, skipping write")

    # -------------------------------
    # LOAD MAPPING FILE
    # -------------------------------
    df_insurername = pd.read_excel(
        mapping_path,
        engine="openpyxl",
        keep_default_na=False
    )

    return final_df, df_insurername, log_df




base_folder = r"\\10.0.11.37\bd_pbp\DB Upload\Revenue_cal_table Update"
log_path = r"\\10.0.11.37\bd_pbp\DB Upload\Revenue_file_ingestion\ingestion_log.xlsx"
mapping_path = r"\\10.0.11.37\bd_pbp\DB Upload\Revenue_file_ingestion\Insurer name Vs db_updated.xlsx"

valid_files=get_today_files(base_folder)

# if valid_files:

# print()
print(valid_files)


if not valid_files:
    print("No files found today. Exiting.")
    sys.exit(0)


df, df_insurername, log_df = ingest_files(valid_files, log_path, mapping_path)
print(df.head())
print(log_df.head())




Nov'26
\\10.0.11.37\bd_pbp\DB Upload\Revenue_cal_table Update\Nov'26
["58_Health & Travel Oct'26 Revenue.xlsx", "59_Health & Travel Sep'26 Recon...xlsx", 'Additional Booking revenue 12-11.xlsx', 'Additional Case 10-11-2026 format.xlsx', 'Additional case 13-11-2026 format.xlsx', 'Additional revenue 10-11-2026.xlsx', 'Additional_10-Nov.xlsx', 'Additional_11-Nov.xlsx', 'Additional_Cases_10-11-2026.xlsx', 'Bajaj_Correction_11-2026.xlsx', 'Bajaj_Corr_12-11-2026.xlsx', 'Bajaj_Tw_Correction-12-11-2026.xlsx', "Commercial Lines_ Revenue Update - 03 Nov'26.xlsx", "Commercial Lines_ Revenue Update - 11th Nov'26.xlsx", "Commercial Lines_ Revenue Update - 13th Nov'26.xlsx", "Commercial Lines_ Revenue Update - 4th Nov'26.xlsx", "Commercial Lines_ Revenue Update - Oct'26.xlsx", "Compile revenue Oct'26 Format.xlsx", "Compile Revenue PSU Oct'26 format.xlsx", 'Digit Correction 18-11.xlsx', "Digit Jul'26 Correction.xlsx", 'Digit_Correction_02-11-2026.xlsx', 'Indudind_Additional_11-11-2026.xlsx', 'Kotak C

In [24]:

import pandas as pd
from sqlalchemy import text
import unicodedata
import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

drivers = pyodbc.drivers()
print(drivers)

['SQL Server', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [ ]:
from rapidfuzz import process, fuzz, utils
import pandas as pd
import re
mapping_path = r"\\10.0.11.37\bd_pbp\DB Upload\Revenue_file_ingestion\Insurer name Vs db_updated.xlsx"

df_insurername = pd.read_excel(
    mapping_path,
    engine="openpyxl",
    keep_default_na=False
)

mapping_df = df_insurername[["InsurerName", "new insurer name"]]

choices = mapping_df["InsurerName"].str.lower().tolist()
print()
# def get_best_match(name):
#     match = process.extractOne(
#         name,
#         choices,
#         scorer=fuzz.token_sort_ratio,
#         score_cutoff=80
#     )
#     if match:
#         return match[0], match[1]
#     return None, None

# unmatched=pd.Series(['Shriram insurance','GI insurance'])
# result=[]
# for name in unmatched:
#     i,k=get_best_match(name.lower())
#     result.append({i:k})
# print(result)    



def get_best_match(name):
    match = process.extractOne(
        name,
        choices,
        scorer=fuzz.token_set_ratio,                 # or fuzz.token_set_ratio
        processor=utils.default_process,    # lowercases + cleans both sides
        score_cutoff=80
    )
    print(match)
    print('#'*20)
    if match:
        choice, score = match
        return choice, mapping_df["new insurer name"].iloc[idx], score
    return None, None, None

unmatched=pd.Series(['Shriram insurance','Kshema Limited','TATA GI insurance'])
result = []
for name in unmatched:
    matched, new_name, score = get_best_match(name)
    result.append({"input": name, "matched": matched, "new_name": new_name, "score": score})

print(pd.DataFrame(result))

# NOISE = {"ltd", "limited", "co", "company", "comp", "insurance", "general",
#          "assurance", "pvt", "private", "inc", "corp", "the", "of", "and"}

# def normalize(s):
#     s = re.sub(r"[^\w\s]", " ", str(s).lower())   # drop punctuation
#     tokens = [t for t in s.split() if t not in NOISE]
#     return " ".join(tokens)

# mapping_df = df_insurername[["InsurerName", "new insurer name"]].copy()
# mapping_df["norm"] = mapping_df["InsurerName"].map(normalize)
# choices = mapping_df["norm"].tolist()

# def get_best_match(name, cutoff=88):
#     q = normalize(name)
#     if not q:
#         return None, None, None
#     match = process.extractOne(
#         q, choices,
#         scorer=fuzz.token_sort_ratio,   # strict now that noise is removed
#         score_cutoff=cutoff
#     )
#     if match:
#         _, score, idx = match
#         row = mapping_df.iloc[idx]
#         return row["InsurerName"], row["new insurer name"], score
#     return None, None, None





# NOISE = {"ltd", "limited", "co", "company", "comp", "insurance", "general",
#          "assurance", "pvt", "private", "inc", "corp", "the", "of", "and"}


# unmatched=pd.Series(['Shriram insurance','GI insurance'])
# def normalize(s):
#     s = re.sub(r"[^\w\s]", " ", str(s).lower())   # drop punctuation
#     tokens = [t for t in s.split() if t not in NOISE]
#     return " ".join(tokens)

# for name in unmatched:
#     q = normalize(name)
#     print(q)











('shriram general insurance company ltd', 100.0, 45)
####################
('kshema general insurance limited', 100.0, 434)
####################
('tata aig general insurance company limited', 90.3225806451613, 2)
####################
               input                                     matched  \
0  Shriram insurance       shriram general insurance company ltd   
1     Kshema Limited            kshema general insurance limited   
2  TATA GI insurance  tata aig general insurance company limited   

                                     new_name       score  
0          Shriram General Insurance Co. Ltd.  100.000000  
1            Kshema General Insurance Limited  100.000000  
2  Tata AIG General Insurance Company Limited   90.322581  


In [ ]:
unmatched=pd.Series(['Shriram insurance'])


0    Shriram insurance
dtype: str


In [1]:
print('hello mohit')

hello mohit
